# 基本面锚定反转完整教程：从传统反转到 FAR

## 📚 教学目标
1. 理解**短期反转异象**的定义：过去 1 个月累计收益率预测未来收益
2. 掌握**基本面锚定反转 (FAR)** 的核心思想：剥离基本面影响后的反转
3. 在微型数据集（10 只股票）上**手算**反转信号和 F-Score 结合
4. 扩展到 300 只股票 × 60 个月，比较**传统反转 vs FAR** 的表现
5. 理解反转效应与动量效应的互补关系

**参考书**：《因子投资：方法与实践》（石川）第 5.2 节
**教学策略**：先用极小数据集让你"看见"每一步计算，再扩展到真实规模

---

## 1. 什么是反转异象？为什么需要基本面锚定？

### 🎯 一个直觉问题

假设你发现一只股票上个月跌了 20%。你会买入吗？

- **情况 A**：市场过度反应，股价会反弹 → 买入赚钱
- **情况 B**：公司基本面恶化（如业绩暴雷），股价会继续跌 → 买入亏钱

**传统反转策略无法区分这两种情况！**

### 📖 书中的定义

> 股票的超跌主要有三个原因：（1）上市公司基本面恶化；（2）投资者对信息的过度反应；（3）噪音交易者导致的瞬时流动性冲击。在上述三个原因中，由于基本面恶化造成的股价大幅下跌往往难以反转。

### 💡 核心洞察

| 概念 | 含义 |
|------|------|
| 短期反转 | 过去 1 个月跌幅大的股票，未来收益更高 |
| 传统反转 | 直接按过去收益率排序，包含基本面恶化的影响 |
| 基本面锚定反转 (FAR) | 剥离基本面影响后的反转，只捕捉过度反应和流动性冲击 |
| 核心公式 | Residual = R - R_FScore |

In [ ]:
import sys, os
print(f"Python: {sys.executable}")
print(f"Version: {sys.version}")
try:
    import matplotlib
    print(f"matplotlib: {matplotlib.__version__}")
except ImportError:
    print("❌ matplotlib 未安装! 请运行: !pip install matplotlib seaborn statsmodels scipy")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

# 设置中文字体和样式
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 基本面锚定反转的金融学依据

### 📐 核心模型

Da et al. (2014) 提出的模型：

$$\text{Residual}_{t+1} = R_{t+1} - \mu_t - \text{CF}_{t+1}$$

其中：
- $R_{t+1}$ = 股票收益率
- $\mu_t$ = 均衡状态下的条件预期收益率
- $\text{CF}_{t+1}$ = 现金流信息造成的收益率变化

### 📐 Zhu et al. (2019) 的改进

使用 F-Score 代替现金流信息：

$$\text{Residual}_{t+1} = R_{t+1} - R_{\text{F-Score}}$$

### 💡 直觉解释

```
传统反转：做多过去输家，做空过去赢家
      ↓
问题：输家中包含基本面恶化的股票（难以反转）
      ↓
改进：用 F-Score 剥离基本面影响
      ↓
FAR：做多基本面好的输家，做空基本面差的赢家
```

### 📐 表 5.5: 短期历史收益率以及 F-Score 叠加后和预期收益率的关系

| | 低 F-Score | 高 F-Score |
|---|---|---|
| **过去输家** | 中等收益 | **高收益**（被低估） |
| **过去赢家** | **低收益**（被高估） | 中等收益 |

---

## 3. 微型示例：10 只股票的反转信号

### 🎯 场景

假设市场上只有 **10 只股票**，我们已知它们上个月的收益率和 F-Score。

我们要：
1. 计算传统反转信号（过去 1 个月收益率）
2. 计算 F-Score 隐含收益率
3. 计算基本面锚定反转信号（Residual）
4. 比较三种策略的效果

In [ ]:
# ========== 构造 10 只股票的微型数据 ==========
np.random.seed(42)

stocks = ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10']

# 上个月收益率（%）：有正有负
past_returns = np.array([-15, -10, -5, -2, 0, 3, 5, 10, 15, 25])

# F-Score（0-9 分）
fscore = np.array([8, 6, 4, 2, 7, 3, 5, 9, 1, 4])

# F-Score 隐含收益率（模拟：F-Score 每增加 1 分，预期收益增加 0.5%）
gamma_fscore = 0.5
r_fscore = gamma_fscore * fscore

# 基本面锚定反转信号 = 过去收益率 - F-Score 隐含收益率
far_signal = past_returns - r_fscore

# 未来收益率（模拟）
# 真实效应：传统反转效应 + F-Score 效应
gamma_reversal = -0.15  # 过去收益率每增加 1%，未来收益减少 0.15%
gamma_f = 0.3           # F-Score 每增加 1 分，未来收益增加 0.3%
noise = np.random.normal(0, 2, 10)
future_returns = 5 + gamma_reversal * past_returns + gamma_f * fscore + noise

# 构建 DataFrame
df_micro = pd.DataFrame({
    'Stock': stocks,
    'Past Return (%)': past_returns,
    'F-Score': fscore,
    'R_FScore (%)': np.round(r_fscore, 2),
    'FAR Signal': np.round(far_signal, 2),
    'Future Return (%)': np.round(future_returns, 2)
})

print("📋 10 只股票数据集：")
print(df_micro.to_string(index=False))
print(f"\n💡 观察：")
print(f"   传统反转信号（Past Return）：S01 跌最多（-15%），S10 涨最多（25%）")
print(f"   F-Score：S08 最高（9 分），S09 最低（1 分）")
print(f"   FAR Signal = Past Return - R_FScore，剥离了基本面影响")

In [ ]:
# ========== 步骤 1: 传统反转策略 ==========
print("📊 步骤 1: 传统反转策略（按过去收益率排序）")
print("─" * 55)

df_sorted = df_micro.sort_values('Past Return (%)').reset_index(drop=True)

# 分为 2 组：Loser (低收益) vs Winner (高收益)
df_sorted['Reversal_Group'] = ['Loser'] * 5 + ['Winner'] * 5

print("\n  Loser 组（过去输家，做多）:")
for _, row in df_sorted[df_sorted['Reversal_Group'] == 'Loser'].iterrows():
    print(f"    {row['Stock']}: Past Ret = {row['Past Return (%)']:>5}%, F-Score = {row['F-Score']}, Future Ret = {row['Future Return (%)']:.2f}%")

print("\n  Winner 组（过去赢家，做空）:")
for _, row in df_sorted[df_sorted['Reversal_Group'] == 'Winner'].iterrows():
    print(f"    {row['Stock']}: Past Ret = {row['Past Return (%)']:>5}%, F-Score = {row['F-Score']}, Future Ret = {row['Future Return (%)']:.2f}%")

# 计算 Spread
ret_loser = df_sorted[df_sorted['Reversal_Group'] == 'Loser']['Future Return (%)'].mean()
ret_winner = df_sorted[df_sorted['Reversal_Group'] == 'Winner']['Future Return (%)'].mean()
spread_traditional = ret_loser - ret_winner

print(f"\n📊 步骤 2: 计算传统反转 Spread")
print(f"  R_Loser = {ret_loser:.2f}%")
print(f"  R_Winner = {ret_winner:.2f}%")
print(f"  Spread = R_Loser - R_Winner = {ret_loser:.2f} - {ret_winner:.2f} = {spread_traditional:.2f}%")
print(f"\n💡 传统反转策略：做多输家、做空赢家，Spread = {spread_traditional:.2f}%")

In [ ]:
# ========== 步骤 3: 基本面锚定反转策略 ==========
print("📊 步骤 3: 基本面锚定反转策略（按 FAR Signal 排序）")
print("─" * 55)

df_far = df_micro.sort_values('FAR Signal').reset_index(drop=True)

# 分为 2 组：Low FAR (被低估) vs High FAR (被高估)
df_far['FAR_Group'] = ['Low FAR'] * 5 + ['High FAR'] * 5

print("\n  Low FAR 组（被低估，做多）:")
for _, row in df_far[df_far['FAR_Group'] == 'Low FAR'].iterrows():
    print(f"    {row['Stock']}: FAR Signal = {row['FAR Signal']:>6.2f}, Past Ret = {row['Past Return (%)']:>5}%, F-Score = {row['F-Score']}, Future Ret = {row['Future Return (%)']:.2f}%")

print("\n  High FAR 组（被高估，做空）:")
for _, row in df_far[df_far['FAR_Group'] == 'High FAR'].iterrows():
    print(f"    {row['Stock']}: FAR Signal = {row['FAR Signal']:>6.2f}, Past Ret = {row['Past Return (%)']:>5}%, F-Score = {row['F-Score']}, Future Ret = {row['Future Return (%)']:.2f}%")

# 计算 Spread
ret_low_far = df_far[df_far['FAR_Group'] == 'Low FAR']['Future Return (%)'].mean()
ret_high_far = df_far[df_far['FAR_Group'] == 'High FAR']['Future Return (%)'].mean()
spread_far = ret_low_far - ret_high_far

print(f"\n📊 步骤 4: 计算 FAR Spread")
print(f"  R_Low FAR = {ret_low_far:.2f}%")
print(f"  R_High FAR = {ret_high_far:.2f}%")
print(f"  Spread = R_Low FAR - R_High FAR = {ret_low_far:.2f} - {ret_high_far:.2f} = {spread_far:.2f}%")
print(f"\n💡 FAR 策略：做多基本面好的输家、做空基本面差的赢家，Spread = {spread_far:.2f}%")

In [ ]:
# ========== 步骤 5: 双重排序：Past Return × F-Score ==========
print("📊 步骤 5: 双重排序：Past Return × F-Score")
print("─" * 55)

# 按 Past Return 分 2 组
df_micro['Rev_Group'] = pd.qcut(df_micro['Past Return (%)'], 2, labels=['Loser', 'Winner'])

# 按 F-Score 分 2 组
df_micro['F_Group'] = pd.cut(df_micro['F-Score'], 
                              bins=[-1, 4, 9], 
                              labels=['Low F', 'High F'])

# 交叉分组
cross_table = df_micro.groupby(['Rev_Group', 'F_Group'])['Future Return (%)'].agg(['mean', 'count'])
print("\n  双重排序结果（平均收益 %）：")
print(cross_table.to_string())

# 计算 FAR 和 FUR
far_portfolio = df_micro[(df_micro['Rev_Group'] == 'Loser') & (df_micro['F_Group'] == 'High F')]
fur_portfolio = df_micro[(df_micro['Rev_Group'] == 'Loser') & (df_micro['F_Group'] == 'Low F')]

print(f"\n💡 核心发现：")
print(f"   FAR（基本面好的输家）: 平均收益 = {far_portfolio['Future Return (%)'].mean():.2f}%")
print(f"   FUR（基本面差的输家）: 平均收益 = {fur_portfolio['Future Return (%)'].mean():.2f}%")
print(f"   差异 = {far_portfolio['Future Return (%)'].mean() - fur_portfolio['Future Return (%)'].mean():.2f}%")
print(f"\n🎯 结论：基本面好的输家（被低估）比基本面差的输家（价值陷阱）收益更高！")

In [ ]:
# ========== 可视化：反转信号对比 ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- 图1: Past Return vs Future Return ---
ax1 = axes[0]
colors = ['#e74c3c' if pr < 0 else '#2ecc71' for pr in df_micro['Past Return (%)']]
ax1.scatter(df_micro['Past Return (%)'], df_micro['Future Return (%)'], 
            c=colors, s=100, edgecolors='black', alpha=0.8)

# 标注股票名
for i, row in df_micro.iterrows():
    ax1.annotate(row['Stock'], (row['Past Return (%)'], row['Future Return (%)']), 
                 fontsize=9, ha='center', va='bottom')

# 添加趋势线
z = np.polyfit(df_micro['Past Return (%)'], df_micro['Future Return (%)'], 1)
p = np.poly1d(z)
x_line = np.linspace(-20, 30, 100)
ax1.plot(x_line, p(x_line), 'r--', linewidth=2, alpha=0.7, label=f'Trend: y={z[0]:.2f}x+{z[1]:.2f}')

ax1.set_xlabel('Past Return (%)', fontsize=12)
ax1.set_ylabel('Future Return (%)', fontsize=12)
ax1.set_title('Traditional Reversal: Past vs Future Returns', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.axvline(x=0, color='black', linewidth=0.8)

# --- 图2: FAR Signal vs Future Return ---
ax2 = axes[1]
scatter = ax2.scatter(df_micro['FAR Signal'], df_micro['Future Return (%)'], 
                      c=df_micro['F-Score'], cmap='RdYlGn', s=100, 
                      edgecolors='black', alpha=0.8, vmin=0, vmax=9)
plt.colorbar(scatter, ax=ax2, label='F-Score')

# 标注股票名
for i, row in df_micro.iterrows():
    ax2.annotate(row['Stock'], (row['FAR Signal'], row['Future Return (%)']), 
                 fontsize=9, ha='center', va='bottom')

ax2.set_xlabel('FAR Signal (Past Return - R_FScore)', fontsize=12)
ax2.set_ylabel('Future Return (%)', fontsize=12)
ax2.set_title('FAR Signal vs Future Returns (colored by F-Score)', fontsize=14, fontweight='bold')
ax2.grid(alpha=0.3)
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.axvline(x=0, color='black', linewidth=0.8)

# --- 图3: 双重排序热力图 ---
ax3 = axes[2]
cross_matrix = df_micro.groupby(['Rev_Group', 'F_Group'])['Future Return (%)'].mean().unstack()
sns.heatmap(cross_matrix, annot=True, fmt='.2f', cmap='RdYlGn', 
            linewidths=1, ax=ax3, cbar_kws={'label': 'Future Return (%)'})
ax3.set_xlabel('F-Score Group', fontsize=12)
ax3.set_ylabel('Past Return Group', fontsize=12)
ax3.set_title('Double Sort: Past Return × F-Score', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"   左图：传统反转，过去收益率与未来收益负相关")
print(f"   中图：FAR Signal 与未来收益的关系，颜色越绿（F-Score 越高）收益越高")
print(f"   右图：双重排序热力图，Loser+High F-Score 组合收益最高")

---

## 4. 完整流程：300 只股票 × 60 个月

### 🎯 完整异象检验

现在我们**从头开始**构建完整的反转异象检验：

1. 模拟 300 只股票、60 个月的截面数据
2. 每月计算传统反转信号和 FAR 信号
3. 按信号排序 → 构建多空组合
4. 收集 60 个月的 Spread
5. 将 Spread 对 FF3 做回归 → 检验 Alpha

### 📐 DGP (数据生成过程)

$$r_{i,t} = \beta_{\text{MKT},i} \cdot \text{MKT}_t + \beta_{\text{SMB},i} \cdot \text{SMB}_t + \beta_{\text{HML},i} \cdot \text{HML}_t + \gamma_{\text{rev}} \cdot \text{PastRet}_{i,t} + \gamma_{\text{F}} \cdot \text{F-Score}_{i,t} + \varepsilon_{i,t}$$

其中：
- $\text{MKT}_t \sim N(0.5, 4^2)$, $\text{SMB}_t \sim N(0.2, 2^2)$, $\text{HML}_t \sim N(0.3, 2^2)$
- $\gamma_{\text{rev}} = -0.15$（反转效应：过去收益每增加 1%，未来收益减少 0.15%）
- $\gamma_{\text{F}} = 0.3$（F-Score 效应：每增加 1 分，未来收益增加 0.3%）
- $\varepsilon_{i,t} \sim N(0, 5^2)$（个股噪声）

In [ ]:
# ========== 完整 DGP: 300 只股票 × 60 个月 ==========
np.random.seed(42)
N_STOCKS = 300
N_MONTHS = 60
N_GROUPS = 5

# FF3 因子月度收益
MKT_f = np.random.normal(0.5, 4, N_MONTHS)
SMB_f = np.random.normal(0.2, 2, N_MONTHS)
HML_f = np.random.normal(0.3, 2, N_MONTHS)

# 每只股票的因子暴露（固定不变）
beta_mkt_i = np.random.uniform(0.5, 1.5, N_STOCKS)
beta_smb_i = np.random.uniform(-0.5, 0.5, N_STOCKS)
beta_hml_i = np.random.uniform(-0.5, 0.5, N_STOCKS)

# 反转效应和 F-Score 效应系数
gamma_reversal = -0.15  # 反转效应
gamma_fscore = 0.3      # F-Score 效应

# 存储每月的 Spread
traditional_spreads = []  # 传统反转
far_spreads = []          # 基本面锚定反转
fur_spreads = []          # 非基本面锚定反转

print(f"📊 模拟参数：")
print(f"  股票数量: {N_STOCKS} 只/月")
print(f"  时间跨度: {N_MONTHS} 个月")
print(f"  分组数量: {N_GROUPS} 组")
print(f"  反转效应: γ_rev = {gamma_reversal}")
print(f"  F-Score 效应: γ_F = {gamma_fscore}")
print(f"")
print(f"🔄 开始逐月模拟...")

for t in range(N_MONTHS):
    # 上个月收益率（截面变异）
    past_ret = np.random.normal(0, 10, N_STOCKS)  # 上月收益 ~ N(0, 10%)
    
    # F-Score
    fscore = np.random.binomial(9, 0.5, N_STOCKS)  # 0-9 分
    
    # 噪声
    eps = np.random.normal(0, 5, N_STOCKS)
    
    # 收益率 = 因子暴露 × 因子收益 + 反转效应 + F-Score 效应 + 噪声
    ret = (beta_mkt_i * MKT_f[t] + beta_smb_i * SMB_f[t]
           + beta_hml_i * HML_f[t] + gamma_reversal * past_ret + gamma_fscore * fscore + eps)
    
    # --- 策略 1: 传统反转 ---
    df_t = pd.DataFrame({'past_ret': past_ret, 'fscore': fscore, 'ret': ret})
    df_t['rev_group'] = pd.qcut(df_t['past_ret'], N_GROUPS, labels=[f'Q{i}' for i in range(1, N_GROUPS+1)])
    rev_group_ret = df_t.groupby('rev_group')['ret'].mean()
    spread_trad = rev_group_ret['Q1'] - rev_group_ret['Q5']  # Loser - Winner
    traditional_spreads.append(spread_trad)
    
    # --- 策略 2: 基本面锚定反转 (FAR) ---
    # FAR Signal = Past Return - R_FScore
    r_fscore = gamma_fscore * fscore
    far_signal = past_ret - r_fscore
    df_t['far_signal'] = far_signal
    
    # 双重排序：Past Return × F-Score
    df_t['rev_group2'] = pd.qcut(df_t['past_ret'], N_GROUPS, labels=[f'Q{i}' for i in range(1, N_GROUPS+1)])
    df_t['f_group'] = pd.cut(df_t['fscore'], bins=[-1, 4, 9], labels=['Low F', 'High F'])
    
    # FAR: 做多 Loser+High F，做空 Winner+Low F
    loser_high_f = df_t[(df_t['rev_group2'] == 'Q1') & (df_t['f_group'] == 'High F')]['ret'].mean()
    winner_low_f = df_t[(df_t['rev_group2'] == 'Q5') & (df_t['f_group'] == 'Low F')]['ret'].mean()
    spread_far = loser_high_f - winner_low_f
    far_spreads.append(spread_far)
    
    # FUR: 做多 Loser+Low F，做空 Winner+High F
    loser_low_f = df_t[(df_t['rev_group2'] == 'Q1') & (df_t['f_group'] == 'Low F')]['ret'].mean()
    winner_high_f = df_t[(df_t['rev_group2'] == 'Q5') & (df_t['f_group'] == 'High F')]['ret'].mean()
    spread_fur = loser_low_f - winner_high_f
    fur_spreads.append(spread_fur)
    
    if t < 3:
        print(f"\n  月份 {t+1}:")
        print(f"    传统反转 Spread (Loser-Winner) = {spread_trad:.2f}%")
        print(f"    FAR Spread (Loser/High F - Winner/Low F) = {spread_far:.2f}%")
        print(f"    FUR Spread (Loser/Low F - Winner/High F) = {spread_fur:.2f}%")

traditional_spreads = np.array(traditional_spreads)
far_spreads = np.array(far_spreads)
fur_spreads = np.array(fur_spreads)

print(f"\n  ... (省略第 4~{N_MONTHS} 月) ...")
print(f"\n✅ 完成！")
print(f"  传统反转 Spread 均值: {traditional_spreads.mean():.3f}%")
print(f"  FAR Spread 均值: {far_spreads.mean():.3f}%")
print(f"  FUR Spread 均值: {fur_spreads.mean():.3f}%")

In [ ]:
# ========== 异象检验: 对 FF3 做回归 ==========
print("═" * 60)
print("📋 异象检验: 回归 Spread 对 FF3 因子")
print("═" * 60)

# 构建 FF3 自变量矩阵
X_ff3 = sm.add_constant(np.column_stack([MKT_f, SMB_f, HML_f]))
factor_names = ['Alpha', 'MKT', 'SMB', 'HML']

# ---- 策略 1: 传统反转 ----
print("\n" + "─" * 60)
print("📊 检验 1: 传统反转")
print("─" * 60)

model_trad = sm.OLS(traditional_spreads, X_ff3).fit()

print(f"\n  {'参数':>10s}  {'估计值':>10s}  {'标准误':>10s}  {'t值':>10s}  {'P值':>10s}  {'显著?':>6s}")
print(f"  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*6}")
for name, coef, se, tv, pv in zip(factor_names, model_trad.params, model_trad.bse,
                                   model_trad.tvalues, model_trad.pvalues):
    sig = '✅' if abs(tv) > 2 else ''
    print(f"  {name:>10s}  {coef:>10.4f}  {se:>10.4f}  {tv:>10.4f}  {pv:>10.6f}  {sig:>6s}")

print(f"\n  R² = {model_trad.rsquared:.4f}")
print(f"  🎯 Alpha = {model_trad.params[0]:.4f}%, t = {model_trad.tvalues[0]:.4f}")
if abs(model_trad.tvalues[0]) > 2:
    print(f"  ✅ Alpha 显著 → 传统反转构成异象")
else:
    print(f"  ❌ Alpha 不显著 → 传统反转不构成异象")

# ---- 策略 2: FAR ----
print("\n" + "─" * 60)
print("📊 检验 2: 基本面锚定反转 (FAR)")
print("─" * 60)

model_far = sm.OLS(far_spreads, X_ff3).fit()

print(f"\n  {'参数':>10s}  {'估计值':>10s}  {'标准误':>10s}  {'t值':>10s}  {'P值':>10s}  {'显著?':>6s}")
print(f"  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*6}")
for name, coef, se, tv, pv in zip(factor_names, model_far.params, model_far.bse,
                                   model_far.tvalues, model_far.pvalues):
    sig = '✅' if abs(tv) > 2 else ''
    print(f"  {name:>10s}  {coef:>10.4f}  {se:>10.4f}  {tv:>10.4f}  {pv:>10.6f}  {sig:>6s}")

print(f"\n  R² = {model_far.rsquared:.4f}")
print(f"  🎯 Alpha = {model_far.params[0]:.4f}%, t = {model_far.tvalues[0]:.4f}")
if abs(model_far.tvalues[0]) > 2:
    print(f"  ✅ Alpha 显著 → FAR 构成异象")
else:
    print(f"  ❌ Alpha 不显著 → FAR 不构成异象")

# ---- 策略 3: FUR ----
print("\n" + "─" * 60)
print("📊 检验 3: 非基本面锚定反转 (FUR)")
print("─" * 60)

model_fur = sm.OLS(fur_spreads, X_ff3).fit()

print(f"\n  {'参数':>10s}  {'估计值':>10s}  {'标准误':>10s}  {'t值':>10s}  {'P值':>10s}  {'显著?':>6s}")
print(f"  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*10}  {'─'*6}")
for name, coef, se, tv, pv in zip(factor_names, model_fur.params, model_fur.bse,
                                   model_fur.tvalues, model_fur.pvalues):
    sig = '✅' if abs(tv) > 2 else ''
    print(f"  {name:>10s}  {coef:>10.4f}  {se:>10.4f}  {tv:>10.4f}  {pv:>10.6f}  {sig:>6s}")

print(f"\n  R² = {model_fur.rsquared:.4f}")
print(f"  🎯 Alpha = {model_fur.params[0]:.4f}%, t = {model_fur.tvalues[0]:.4f}")
if abs(model_fur.tvalues[0]) > 2:
    print(f"  ✅ Alpha 显著 → FUR 构成异象")
else:
    print(f"  ❌ Alpha 不显著 → FUR 不构成异象")

In [ ]:
# ========== 可视化: 三种策略对比 ==========
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
months = np.arange(1, N_MONTHS + 1)

# --- 图1: 传统反转 Spread 时间序列 ---
ax1 = axes[0, 0]
ax1.bar(months, traditional_spreads,
        color=['#2ecc71' if s > 0 else '#e74c3c' for s in traditional_spreads],
        alpha=0.7, edgecolor='none')
ax1.axhline(y=traditional_spreads.mean(), color='blue', linestyle='--', linewidth=2,
            label=f'Mean = {traditional_spreads.mean():.2f}%')
ax1.axhline(y=0, color='black', linewidth=0.8)
ax1.set_xlabel('Month', fontsize=11)
ax1.set_ylabel('Spread (%)', fontsize=11)
ax1.set_title('Traditional Reversal: Monthly Spread', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# --- 图2: FAR Spread 时间序列 ---
ax2 = axes[0, 1]
ax2.bar(months, far_spreads,
        color=['#2ecc71' if s > 0 else '#e74c3c' for s in far_spreads],
        alpha=0.7, edgecolor='none')
ax2.axhline(y=far_spreads.mean(), color='blue', linestyle='--', linewidth=2,
            label=f'Mean = {far_spreads.mean():.2f}%')
ax2.axhline(y=0, color='black', linewidth=0.8)
ax2.set_xlabel('Month', fontsize=11)
ax2.set_ylabel('Spread (%)', fontsize=11)
ax2.set_title('Fundamental-Anchored Reversal (FAR): Monthly Spread', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3)

# --- 图3: FUR Spread 时间序列 ---
ax3 = axes[1, 0]
ax3.bar(months, fur_spreads,
        color=['#2ecc71' if s > 0 else '#e74c3c' for s in fur_spreads],
        alpha=0.7, edgecolor='none')
ax3.axhline(y=fur_spreads.mean(), color='blue', linestyle='--', linewidth=2,
            label=f'Mean = {fur_spreads.mean():.2f}%')
ax3.axhline(y=0, color='black', linewidth=0.8)
ax3.set_xlabel('Month', fontsize=11)
ax3.set_ylabel('Spread (%)', fontsize=11)
ax3.set_title('Fundamental-Unanchored Reversal (FUR): Monthly Spread', fontsize=13, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(axis='y', alpha=0.3)

# --- 图4: Alpha 对比 ---
ax4 = axes[1, 1]
strategies = ['Traditional', 'FAR', 'FUR']
alphas = [model_trad.params[0], model_far.params[0], model_fur.params[0]]
t_values = [model_trad.tvalues[0], model_far.tvalues[0], model_fur.tvalues[0]]
colors_alpha = ['#2ecc71' if abs(t) > 2 else '#95a5a6' for t in t_values]

bars = ax4.bar(strategies, alphas, color=colors_alpha, edgecolor='black', alpha=0.85, width=0.5)
ax4.axhline(y=0, color='red', linestyle='--', linewidth=1.5, label='H₀: α=0')

# 标注 t 值
for bar, a, tv in zip(bars, alphas, t_values):
    va = 'bottom' if a >= 0 else 'top'
    offset = 0.05 if a >= 0 else -0.05
    ax4.text(bar.get_x() + bar.get_width()/2., a + offset,
             f'α={a:.2f}%\nt={tv:.2f}', ha='center', va=va, fontweight='bold', fontsize=10)

ax4.set_ylabel('Alpha (%)', fontsize=12)
ax4.set_title('Alpha Comparison: Three Strategies', fontsize=13, fontweight='bold')
ax4.legend(fontsize=10)
ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  上排：传统反转和 FAR 的月度 Spread 时间序列")
print(f"  下排左：FUR 的月度 Spread（非基本面锚定）")
print(f"  下排右：三种策略的 Alpha 对比（绿色 = 显著，灰色 = 不显著）")
print(f"  🎯 关键发现：FAR 的 Alpha 通常比传统反转更显著，FUR 可能不显著")

In [ ]:
# ========== 综合对比报告 ==========
print("=" * 60)
print("📋 反转异象检验综合报告")
print("=" * 60)

print(f"\n🎯 研究问题:")
print(f"   短期反转效应是否存在？基本面锚定能否提升反转策略？")

print(f"\n📊 数据概况:")
print(f"   股票数量: {N_STOCKS} 只/月")
print(f"   时间跨度: {N_MONTHS} 个月")
print(f"   分组方式: {N_GROUPS} 分位，Loser-Winner Spread")

print(f"\n📊 Spread 统计:")
print(f"   {'策略':>15s}  {'Spread均值':>12s}  {'Spread标准差':>14s}  {'Sharpe Ratio':>14s}")
print(f"   {'─'*15}  {'─'*12}  {'─'*14}  {'─'*14}")

sr_trad = traditional_spreads.mean() / traditional_spreads.std(ddof=1) * np.sqrt(12)
sr_far = far_spreads.mean() / far_spreads.std(ddof=1) * np.sqrt(12)
sr_fur = fur_spreads.mean() / fur_spreads.std(ddof=1) * np.sqrt(12)

print(f"   {'传统反转':>15s}  {traditional_spreads.mean():>12.3f}%  {traditional_spreads.std(ddof=1):>14.3f}%  {sr_trad:>14.3f}")
print(f"   {'FAR':>15s}  {far_spreads.mean():>12.3f}%  {far_spreads.std(ddof=1):>14.3f}%  {sr_far:>14.3f}")
print(f"   {'FUR':>15s}  {fur_spreads.mean():>12.3f}%  {fur_spreads.std(ddof=1):>14.3f}%  {sr_fur:>14.3f}")

print(f"\n🧮 Alpha 检验 (控制 FF3 因子):")
print(f"   {'策略':>15s}  {'α̂':>10s}  {'t_α':>10s}  {'P值':>12s}  {'结论':>10s}")
print(f"   {'─'*15}  {'─'*10}  {'─'*10}  {'─'*12}  {'─'*10}")

for name, model in [('传统反转', model_trad), ('FAR', model_far), ('FUR', model_fur)]:
    alpha = model.params[0]
    t_alpha = model.tvalues[0]
    p_alpha = model.pvalues[0]
    conclusion = '✅ 异象' if abs(t_alpha) > 2 else '❌ 非异象'
    print(f"   {name:>15s}  {alpha:>10.4f}%  {t_alpha:>10.4f}  {p_alpha:>12.6f}  {conclusion:>10s}")

print(f"\n🎯 结论:")
print(f"  1. 传统反转在 A 股中显著存在")
print(f"  2. FAR（基本面锚定反转）的 Alpha 通常比传统反转更显著")
print(f"  3. FUR（非基本面锚定反转）的 Alpha 可能不显著")
print(f"  4. 剥离基本面影响后，反转效应更纯粹、更强劲")

print("\n" + "=" * 60)

---

## 📌 核心概念回顾

### 📌 短期反转异象 (Short-Term Reversal)
- **定义**: 过去 1 个月跌幅大的股票，未来收益更高
- **公式**: $R_{\text{Loser}} - R_{\text{Winner}} > 0$
- **含义**: 市场存在短期过度反应，股价会反转
- **局限**: 包含基本面恶化的股票（难以反转）

### 📌 基本面锚定反转 (FAR)
- **定义**: 剥离基本面影响后的反转信号
- **公式**: $\text{Residual} = R_{t+1} - R_{\text{F-Score}}$
- **含义**: 只捕捉过度反应和流动性冲击，排除基本面恶化
- **优势**: Alpha 更显著，策略更稳健

### 📌 F-Score 隐含收益率
- **定义**: 由 F-Score 预测的收益率部分
- **公式**: $R_{\text{F-Score}} = \gamma \times \text{F-Score}$
- **含义**: 反映公司基本面质量对收益的贡献
- **判断标准**: F-Score 越高，预期收益越高

### 📌 双重排序方法
- **定义**: 同时按两个变量排序分组
- **公式**: 按 Past Return 分 5 组 × 按 F-Score 分 3 组 = 15 个组合
- **含义**: 控制一个变量后，检验另一个变量的效应
- **应用**: FAR = Loser/High F - Winner/Low F

### 🔗 完整流程
```
计算过去收益率 → 计算 F-Score → 计算 R_FScore → 计算 Residual
      ↓              ↓              ↓              ↓
   1个月收益      9个指标        γ × F-Score    R - R_FScore
      ↓              ↓              ↓              ↓
   双重排序 → 构建 FAR 组合 → 收集 Spread → 检验 Alpha
```

---

## ❌ 常见误区

### ❌ 误区 1: 反转策略就是买跌得多的股票
**✓ 正确理解**: 直接买跌得多的股票可能买到基本面恶化的公司。必须用 F-Score 等指标筛选基本面好的输家。

### ❌ 误区 2: FAR 和传统反转是一样的
**✓ 正确理解**: FAR 剥离了基本面影响，只捕捉过度反应和流动性冲击。FAR 的 Alpha 通常比传统反转更显著。

### ❌ 误区 3: 反转效应和动量效应是矛盾的
**✓ 正确理解**: 反转效应（短期 1 个月）和动量效应（中期 3-12 个月）在不同时间尺度上共存，是互补的。

### ❌ 误区 4: 双重排序只需要看单变量的效应
**✓ 正确理解**: 双重排序的目的是控制一个变量后，检验另一个变量的独立效应。必须看交叉分组的结果。

### ❌ 误区 5: F-Score 隐含收益率是固定不变的
**✓ 正确理解**: F-Score 隐含收益率的系数 γ 需要通过数据估计，不同市场、不同时期可能不同。